# A kernel from distance

The radial basis kernel, or RBF kernel, compares two points by how far apart they are:

$$
K(\mathbf{x}, \mathbf{z}) = \exp\bigl(-\gamma \|\mathbf{x}-\mathbf{z}\|^2\bigr)
$$

It equals $1$ when $\mathbf{x} = \mathbf{z}$. It shrinks toward $0$ as they move apart. $\gamma > 0$ chooses the speed. There is an infinite feature map behind this formula. You do not need to write that map down to use the kernel.


## Two numbers, computed by hand

Take $\gamma = 1/2$, the point $0$, and the point $2$.

$$
\begin{aligned}
K(0, 0) &= 1 \\
K(2, 2) &= 1 \\
K(0, 2) &= \exp\left(-\frac{1}{2}\cdot 4\right) = e^{-2}
\end{aligned}
$$

$e^{-2}$ is about $0.135$, closer to $0$ than to $1$. These points are far apart on the scale that $\gamma = 1/2$ cares about. Do not replace $e^{-2}$ by a nearby fraction and call it exact.


In [1]:
# Hand values of the RBF kernel. exp(-2) stays exact through math.exp.
import math

gamma = 0.5

def rbf(x, z):
    return math.exp(-gamma * (x - z) ** 2)

assert rbf(0, 0) == 1
assert rbf(2, 2) == 1
assert abs(rbf(0, 2) - math.exp(-2)) < 1e-15
print("K(0, 2)", rbf(0, 2))


K(0, 2) 0.1353352832366127


## A two-point hard margin

Labels: $y(0) = -1$ and $y(2) = +1$. In kernel form, with both points as support vectors and $b = 0$,

$$
f(x) = \alpha\bigl(-K(0, x) + K(2, x)\bigr)
$$

Ask for functional margin $1$ at $x = 0$:

$$
f(0) = \alpha\bigl(-1 + e^{-2}\bigr) = -1
$$

so

$$
\alpha = \frac{1}{1 - e^{-2}}
$$

The same $\alpha$ gives $f(2) = +1$. At the midpoint the two kernels agree, $K(0, 1) = K(2, 1) = e^{-1/2}$, and therefore $f(1) = 0$. The boundary sits halfway between the points, just as a linear SVM would place it, but the score away from the points is no longer a straight line.


In [2]:
# The two-point dual: margin 1 at each training point, score 0 in the middle.
import math

gamma = 0.5
pair = math.exp(-2)
alpha = 1 / (1 - pair)

def rbf(x, z):
    return math.exp(-gamma * (x - z) ** 2)

def score(x):
    return alpha * (-rbf(0, x) + rbf(2, x))

assert abs(score(0) - (-1)) < 1e-12
assert abs(score(2) - 1) < 1e-12
assert abs(score(1)) < 1e-12
# Far to the right the nearer training point dominates, so the score stays positive.
assert score(4) > 0
print("alpha", alpha)
print("score at 0, 1, 2, 4:", score(0), score(1), score(2), score(4))


alpha 1.1565176427496657
score at 0, 1, 2, 4: -1.0 0.0 1.0 0.15612967430201324


## A pitfall

A large $\gamma$ makes $K$ decay in a tiny neighborhood. Training points can become isolated islands, each with its own spike. A small $\gamma$ makes distant points look alike. On two points the boundary stayed in the middle for $\gamma = 1/2$; that will not save a crowded data set if $\gamma$ is chosen carelessly.

## What to carry forward

$K = \exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$. For $\gamma = 1/2$, labels $-1$ at $0$ and $+1$ at $2$, the hard-margin coefficient is $1/(1-e^{-2})$ and the midpoint scores $0$.
